# [5] Detecting LLM-Generated Text with Binoculars

:ref: https://huggingface.co/blog/dmicz/binoculars-text-detection

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import SWUnivDaconDataset

from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from torch.utils.data import DataLoader
from torch.nn import functional as F
from torch import nn
import torch

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import sys
import gc

In [ ]:
for test in tqdm(range(1000), desc="Testing tqdm"):
    pass

In [ ]:
!nvidia-smi

In [ ]:
# Change according to hardware
DEVICE_1 = "cuda:4"
DEVICE_2 = "cuda:5"

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
torch.set_grad_enabled(False)

observer_name = "tiiuae/falcon-7b-instruct"
performer_name = "tiiuae/falcon-7b"

# Verify tokenizers are identical
observer_tokenizer = AutoTokenizer.from_pretrained(observer_name)
performer_tokenizer = AutoTokenizer.from_pretrained(performer_name)

if observer_tokenizer.vocab != performer_tokenizer.vocab:
    raise ValueError("Observer and performer models must have identical tokenizers")

In [ ]:
torch.set_grad_enabled(False)

observer_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
performer_name = "meta-llama/Meta-Llama-3.1-8B"

# Verify tokenizers are identical
observer_tokenizer = AutoTokenizer.from_pretrained(observer_name)
performer_tokenizer = AutoTokenizer.from_pretrained(performer_name)

if observer_tokenizer.vocab != performer_tokenizer.vocab:
    raise ValueError("Observer and performer models must have identical tokenizers")

In [ ]:
torch.set_grad_enabled(False)

observer_name = "google/gemma-3-4b-it-qat-int4-unquantized"
performer_name = "google/gemma-3-4b-pt"

# Verify tokenizers are identical
observer_tokenizer = AutoTokenizer.from_pretrained(observer_name)
performer_tokenizer = AutoTokenizer.from_pretrained(performer_name)

if observer_tokenizer.vocab != performer_tokenizer.vocab:
    raise ValueError("Observer and performer models must have identical tokenizers")

In [ ]:
# Load models with quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model_kwargs = {
    "trust_remote_code": True,
    "torch_dtype": torch.bfloat16,
    "quantization_config": quantization_config
}

In [ ]:
observer_model = AutoModelForCausalLM.from_pretrained(
    observer_name, device_map={"": DEVICE_1}, **model_kwargs
)
observer_model.eval()

In [ ]:
performer_model = AutoModelForCausalLM.from_pretrained(
    performer_name, device_map={"": DEVICE_2}, **model_kwargs
)
performer_model.eval()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(observer_name)
tokenizer.pad_token = tokenizer.eos_token

def tokenize(text):
    return tokenizer(text, return_tensors="pt")

In [ ]:
tokenize("Hello, my dog is cute")

## Perplexity and cross-perplexity

In [ ]:
criterion = nn.CrossEntropyLoss(reduction='none')

In [ ]:
@torch.inference_mode()
def get_logits(encodings):
    observer_logits = observer_model(**encodings.to(DEVICE_1)).logits
    performer_logits = performer_model(**encodings.to(DEVICE_2)).logits
    return observer_logits, performer_logits

In [ ]:
encoding = tokenize('''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
groundbreaking research in astrophysics. With his keen sense of observation and unparalleled ability to interpret
cosmic data, he uncovered new insights into the mysteries of black holes and the origins of the universe. As he
peered through telescopes with his large, round eyes, fellow researchers often remarked that it seemed as if the
stars themselves whispered their secrets directly to him. Dr. Cosmos not only became a beacon of inspiration to
aspiring scientists but also proved that intellect and innovation can be found in the most unexpected of creatures.''')
encoding

In [ ]:
observer_logits, performer_logits = get_logits(encoding)
observer_logits, performer_logits

In [ ]:
encoding.input_ids.shape, observer_logits.shape

In [ ]:
S = observer_logits.shape[-2]
V = observer_logits.shape[-1]

observer_logits[..., :-1, :].contiguous().shape

In [ ]:
encoding.input_ids[..., 1:].shape

In [ ]:
ppl = criterion(observer_logits[..., :-1, :].contiguous().transpose(1, 2).to("cpu"),
                encoding.input_ids[..., 1:].contiguous().to("cpu")).float()

ppl, ppl.sum(1)

In [ ]:
softmax = nn.Softmax(dim=-1)
performer_probs = softmax(performer_logits).view(-1, V)
performer_probs, performer_probs.shape

In [ ]:
observer_scores = observer_logits.view(-1, V).to("cpu")
observer_scores, observer_scores.shape

In [ ]:
xppl = criterion(observer_scores[:-1].to(DEVICE_2), performer_probs[:-1]).view(-1, S - 1)

xppl, xppl.sum(1)

In [ ]:
(ppl.to(DEVICE_2).sum(1) / xppl.sum(1)).cpu().tolist()

In [ ]:
# redefine to handle batch of strings
def tokenize(batch):
    return tokenizer(
        batch, return_tensors="pt", return_token_type_ids=False,
        padding="longest" if len(batch) > 1 else False
    ).to(DEVICE_1)

# redefinition with cuda sync
def get_logits(encodings):
    observer_logits = observer_model(**encodings.to(DEVICE_1)).logits
    performer_logits = performer_model(**encodings.to(DEVICE_2)).logits
    torch.cuda.synchronize()

    return observer_logits, performer_logits

In [ ]:
def perplexity(encoding, logits):
    shifted_logits = logits[..., :-1, :].contiguous().cpu()
    shifted_labels = encoding.input_ids[..., 1:].contiguous().cpu()
    shifted_attention_mask = encoding.attention_mask[..., 1:].contiguous().cpu()

    ppl = criterion(shifted_logits.transpose(1, 2), shifted_labels) * shifted_attention_mask
    ppl = ppl.sum(1) / shifted_attention_mask.sum(1)

    return ppl.cpu().float().numpy()

In [ ]:
def cross_perplexity(observer_logits, performer_logits, encoding):
    V = observer_logits.shape[-1]
    S = observer_logits.shape[-2]

    performer_probs = softmax(performer_logits).view(-1, V).cpu()
    observer_scores = observer_logits.view(-1, V).cpu()

    xppl = criterion(observer_scores, performer_probs).view(-1, S)
    padding_mask = (encoding.input_ids != tokenizer.pad_token_id).type(torch.uint8).cpu()

    xppl = (xppl * padding_mask).sum(1) / padding_mask.sum(1)

    return xppl.cpu().float().numpy()

In [ ]:
def binocular_score(text):
    batch = [text] if isinstance(text, str) else text
    encodings = tokenize(batch)
    observer_logits, performer_logits = get_logits(encodings)
    ppl = perplexity(encodings, observer_logits)
    xppl = cross_perplexity(observer_logits, performer_logits, encodings)

    return (ppl / xppl).tolist()

In [ ]:
tests = ['''The motivation behind LLM Detection is harm reduction, to trace text origins, block spam, and identify fake news produced by LLMs. *Preemptive detection* methods attempt to "watermark" generated text, but requires full control of the generating models, which already seems to be impossible. Therefore, more recent works have been on *post-hoc detection* methods, which could be used without the cooperation of the text's author. The paper's authors suggest that there are two main groups for post-hoc detectors, the first being finetuning a pretrained language model to perform binary classification. There are many additional techniques that make this approach more effective, but all implementations will require training on text produced by the target model, which is both computationally expensive and limited by the number of new models that are being open-sourced.
The second group uses statistical signatures of machine-generated text, with the aim of zero-shot learning. This would allow for the detection of a wide range of models, with little to no training data. These methods use measures such as perplexity, perplexity curvature, log rank, intrinsic dimensionality, and n-gram analysis. The Binoculars paper proposes a focus on low false positive rate (FPR) and high performance on out-of-domain samples, rather than focusing on classifier AUCs for the high-stakes application of LLM detection.''',
 '''Dr. Capy Cosmos, a capybara unlike any other, astounded the scientific community with his
 groundbreaking research in astrophysics. With his keen sense of observation and unparalleled ability to interpret
 cosmic data, he uncovered new insights into the mysteries of black holes and the origins of the universe. As he
 peered through telescopes with his large, round eyes, fellow researchers often remarked that it seemed as if the
 stars themselves whispered their secrets directly to him. Dr. Cosmos not only became a beacon of inspiration to
 aspiring scientists but also proved that intellect and innovation can be found in the most unexpected of creatures.''',
 '''We the People of the United States, in Order to form a more perfect Union, establish Justice, insure domestic Tranquility, provide for the common defence, promote the general Welfare, and secure the Blessings of Liberty to ourselves and our Posterity, do ordain and establish this Constitution for the United States of America.'''
 ]
binocular_score(tests)

## Batch Processing

In [ ]:
sample_amount = 50

df = valid_dataset.raw
df_0 = df[df['generated'] == 0].sample(n=sample_amount, random_state=50)
df_1 = df[df['generated'] == 1].sample(n=sample_amount, random_state=50)
balanced_df = pd.concat([df_0, df_1]).sample(frac=1, random_state=50).reset_index(drop=True)
valid_dataset.data, valid_dataset.labels = balanced_df['full_text'].tolist(), balanced_df['generated'].tolist()
len(valid_dataset)

In [ ]:
BATCH_SIZE = 1, 1, 1

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE[0], shuffle=True)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE[1], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE[2], shuffle=False)

In [ ]:
threshold = 0.88

def flip_scores(scores, threshold=threshold):
    bias = abs(0.5 - threshold)
    length = 0.5 - bias
    if threshold >= 0.5:
        maxval, minval = 1, threshold - length
    else:
        maxval, minval = threshold + length, 0
    return [1- (max(min(score, maxval), minval) - minval) / (maxval-minval) for score in scores]

def to_label(scores, threshold=0.5):
    return [0 if score < threshold else 1 for score in scores]

def accuracy(scores, flipped, preds, labels):
    correct, true_human, false_human = 0, 0, 0
    for score, flip, pred, label in zip(scores, flipped, preds, labels):
        if pred == label:
            if label == 0: true_human += 1
            correct += 1
            print(f"INFO: Correct prediction - expected {label}, got {pred} ({score}->{flip})")
        else:
            if label == 0: false_human += 1
            print(f"ERROR: Incorrect prediction - expected {label}, got {pred} ({score}->{flip})")
    return correct, true_human, false_human

In [ ]:
flip_scores([0.7, 0.8, 0.9])

In [ ]:
with tqdm(valid_loader, desc="[Validating]") as progress:
    corrects, errors, true_human, false_human = 0, 0, 0, 0
    for texts, labels in progress:
        torch.cuda.empty_cache()
        gc.collect()

        try:
            scores = binocular_score(texts)
            flipped = flip_scores(scores)
            predictions = to_label(flipped)

            c, th, fh = accuracy(scores, flipped, predictions, labels.tolist())
            corrects += c
            errors += len(labels) - c
            true_human += th
            false_human += fh
        except Exception as e:
            print(f"ERROR: {e} - {texts}")

        progress.set_description(f"[Validating] Correct: {corrects/(corrects+errors):.6%} [H: {true_human}, A: {corrects-true_human}], Errors: {errors} [H: {false_human}, A: {errors-false_human}]")

In [ ]:
results = []
for texts, labels in tqdm(test_dataset, desc="[Testing]"):
    torch.cuda.empty_cache()
    gc.collect()

    try:
        pred = to_label(flip_scores(binocular_score(texts)))[0]
        results.append(pred)
    except Exception as e:
        print(f"ERROR: {e} - {texts}")
        results.append(0.5)

results

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')
sub

In [ ]:
sub['generated'] = results
sub

In [ ]:
plt.figure(figsize=(12, 7))
sns.histplot(data=sub, x='generated', kde=True, bins=50)
plt.title("Prediction Probability Distribution", fontsize=16)
plt.xlabel("Predicted Probability (Generated = 1)", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.grid(True, linestyle="--", alpha=0.6)

plt.show()
print(sub['generated'].describe())

In [ ]:
sub.to_csv("./data/submission_binoculars.csv", index=False, encoding='utf-8-sig')